## 1. 데이터 준비 및 전체 구조 파악

In [1]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

In [2]:
# 데이터 불러오기
df = pd.read_csv('../dataset/raw/hotel_bookings.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
# 데이터 정보
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

## 2. 결측치 처리

In [4]:
# 결측치 확인
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

### - 변수 종류에 따른 결측치 처리
- children: 자녀수 -----> 0
- country: 국적 -----> 삭제
- agent: 예약을 진행한 여행사의 ID -----> 0: 새 값 추가, 직접 예약
- company: 예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID -----> 0: 새 값 추가, 직접 예약

In [5]:
# 결측치 처리
missing_cols_drop = ['country']
missing_cols_fill = ['children', 'agent', 'company']

df.dropna(subset=missing_cols_drop[0], axis=0, inplace=True)
df[missing_cols_fill] = df[missing_cols_fill].fillna(0)

df.isna().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

## 함수 정의

In [6]:
def drop_target(target):
    df.drop(target, axis=1, inplace=True)

## 3. 도착일 관련 데이터

In [7]:
# 대상: 

# 3   arrival_date_year               119390 non-null  int64

# 도착일 - 년

# 4   arrival_date_month              119390 non-null  str

# 도착일 - 월

# 5   arrival_date_week_number        119390 non-null  int64

# 도착일 - 주차

# 6   arrival_date_day_of_month       119390 non-null  int64

# 도착일 - 일

# 삭제: arrival_date_week_number
cols = ['arrival_date_year', 'arrival_date_day_of_month', 'arrival_date_week_number']
drop_target(cols)

## 4. 투숙객 현황 관련 데이터

9   adults                          119390 non-null  int64

성인 수

10  children                        119386 non-null  float64

자녀 수

11  babies                          119390 non-null  int64

아기 수

26  customer_type                   119390 non-null  str

예약 유형(다음 네 가지 범주 중 하나라고 가정):

- 계약 - 예약에 할당량 또는 기타 유형의 계약이 포함된 경우;
- 그룹 예약 – 예약이 그룹과 연결된 경우;
- 개별 예약 – 단체 예약이나 계약에 포함되지 않고, 다른 개별 예약과 연계되지 않은 예약.
- 일시적 예약 - 예약이 일시적이지만, 적어도 다른 일시적 예약과 연관된 경우

In [8]:
cols = ['adults', 'children', 'babies']
df[cols].describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
adults,118902.0,1.858404,0.578576,0.0,2.0,2.0,2.0,55.0
children,118902.0,0.104203,0.399166,0.0,0.0,0.0,0.0,10.0
babies,118902.0,0.007948,0.097379,0.0,0.0,0.0,0.0,10.0


In [9]:
df.loc[(df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)].shape[0]

170

In [10]:
# 투숙객 유형을 분류하는 함수 정의
def classify_guest_type(row):
    adults = row['adults']
    children = row['children']
    babies = row['babies']
    total_kids = children + babies
    
    # 인원이 모두 0인 경우
    if adults == 0 and total_kids == 0:
        return 'Undefined'
    
    # 아동/유아가 있는 경우 (가족)
    if total_kids > 0:
        return 'Family'
    
    # 성인으로만 구성된 경우
    if adults == 1:
        return 'Single'            # 1인 투숙객
    
    return 'Group'                 # 2인 이상 성인

In [11]:
df['guest_type'] = df[cols].apply(classify_guest_type, axis=1)
df[cols + ['guest_type']].head()

,adults,children,babies,guest_type
0,2,0.0,0,Group
1,2,0.0,0,Group
2,1,0.0,0,Single
3,1,0.0,0,Single
4,2,0.0,0,Group


In [12]:
df['guest_type'].value_counts()

guest_type
Group        87129
Single       22285
Family        9318
Undefined      170
Name: count, dtype: int64

In [13]:
drop_target(cols)

## 5. 객실 유형 관련 데이터

In [14]:
# diff_reserved_room_type 추가: 예약한 방과 배정받은 방이 다른 경우
# reserved_room_type, assigned_room_type 열 조합
# 각 열은 객실 유형을 코드로 나타낸 값으로 실제 객실 유형은 알 수 없으므로 삭제함
cols = ['reserved_room_type', 'assigned_room_type']

df['diff_reserved_room_type'] = (df[cols[0]] != df[cols[1]]).astype(int)

In [15]:
df[cols + ['diff_reserved_room_type']].head()

,reserved_room_type,assigned_room_type,diff_reserved_room_type
0,C,C,0
1,C,C,0
2,A,C,1
3,A,A,0
4,A,A,0


In [16]:
drop_target(cols)

## 예약 유형 관련 데이터
14  market_segment                  119390 non-null  str

시장 부문 명칭. 분류 체계에서 "TA"는 "여행사(Travel Agents)"를, "TO"는 "여행사(Tour Operators)"를 의미합니다.

15  distribution_channel            119390 non-null  str

예약 유통 채널. "TA"는 "여행사"를, "TO"는 "여행 운영업체"를 의미합니다.

23  agent                           103050 non-null  float64

예약을 진행한 여행사의 ID

24  company                         6797 non-null    float64

예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID입니다. 익명성 유지를 위해 직책 대신 ID를 표시합니다.

In [17]:
df['market_segment'].value_counts()

market_segment
Online TA        56403
Offline TA/TO    24160
Groups           19806
Direct           12449
Corporate         5111
Complementary      734
Aviation           237
Undefined            2
Name: count, dtype: int64

In [18]:
df['distribution_channel'].value_counts()

distribution_channel
TA/TO        97730
Direct       14483
Corporate     6491
GDS            193
Undefined        5
Name: count, dtype: int64

In [19]:
df['is_agent'] = np.where(df['agent'] != 0, 1, 0)
df[['agent', 'is_agent']]

,agent,is_agent
0,0.0,0
1,0.0,0
2,0.0,0
3,304.0,1
4,240.0,1
...,...,...
119385,394.0,1
119386,9.0,1
119387,9.0,1
119388,89.0,1


In [20]:
df['is_company'] = np.where(df['company'] != 0, 1, 0)
df[['company', 'is_company']].head()

,company,is_company
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0


In [21]:
df['is_company'].value_counts()

is_company
0    112279
1      6623
Name: count, dtype: int64

In [22]:
drop_target(['agent', 'company'])

## 예약 상태 관련 데이터
30  reservation_status              119390 non-null  str

| 예약 최종 상태는 다음 세 가지 범주 중 하나로 가정합니다. |
| --- |
| 취소됨 – 고객이 예약을 취소했습니다. |
| 체크아웃 - 고객이 체크인은 했지만 이미 출발했습니다. |
| 노쇼 – 고객이 체크인하지 않았으며, 호텔에 그 이유를 알렸습니다. |

31  reservation_status_date         119390 non-null  str

마지막 상태가 설정된 날짜입니다. 이 변수는 *ReservationStatus* 변수와 함께 사용하여 예약이 취소된 시점 또는 고객이 호텔에서 체크아웃한 시점을 파악하는 데 사용할 수 있습니다.

In [23]:
drop_target(['reservation_status', 'reservation_status_date'])

## 기타
정리 필요

In [24]:
df['hotel'].unique()

<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str

In [25]:
df['is_city_hotel'] = (df['hotel'] == 'City Hotel').astype(int)

In [26]:
df['is_city_hotel'].value_counts()

is_city_hotel
1    79306
0    39596
Name: count, dtype: int64

In [27]:
drop_target(['hotel'])

In [28]:
df['meal'].unique()

<StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str

In [29]:
df['meal'].value_counts()

meal
BB           91867
HB           14434
SC           10638
Undefined     1165
FB             798
Name: count, dtype: int64

In [30]:
df['country'].unique()

<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA', 'ROU', 'NOR', 'OMN', 'ARG',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177, dtype: str

In [31]:
# 유럽 국가 코드
europe_codes = [
    'PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'ITA', 'IRL', 'BEL', 'NLD', 'AUT', 
    'POL', 'SWE', 'CHE', 'NOR', 'DNK', 'FIN', 'CZE', 'GRC', 'ROU', 'HUN', 
    'HRV', 'SVK', 'SVN', 'BGR', 'LTU', 'LVA', 'EST', 'LUX', 'MLT', 'CYP'
]

def group_country(country):
    if country == 'PRT':
        return 'PRT'
    elif country in europe_codes:
        return 'EUR'  # 기타 유럽
    else:
        return 'OTH'  # 나머지

df['country_region'] = df['country'].apply(group_country)


In [32]:
df[['country', 'country_region']].head(15)

,country,country_region
0,PRT,PRT
1,PRT,PRT
2,GBR,EUR
3,GBR,EUR
4,GBR,EUR
5,GBR,EUR
6,PRT,PRT
7,PRT,PRT
8,PRT,PRT
9,PRT,PRT


In [33]:
drop_target(['country'])

## 마무리

In [34]:
df.info()

<class 'pandas.DataFrame'>
Index: 118902 entries, 0 to 119389
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   is_canceled                     118902 non-null  int64  
 1   lead_time                       118902 non-null  int64  
 2   arrival_date_month              118902 non-null  str    
 3   stays_in_weekend_nights         118902 non-null  int64  
 4   stays_in_week_nights            118902 non-null  int64  
 5   meal                            118902 non-null  str    
 6   market_segment                  118902 non-null  str    
 7   distribution_channel            118902 non-null  str    
 8   is_repeated_guest               118902 non-null  int64  
 9   previous_cancellations          118902 non-null  int64  
 10  previous_bookings_not_canceled  118902 non-null  int64  
 11  booking_changes                 118902 non-null  int64  
 12  deposit_type                    

In [35]:
# 파일 저장
df.to_csv('../dataset/preprocessed/hotel_bookings.csv', index=False)

In [36]:
dummy_cols = ['arrival_date_month', 'meal', 'market_segment','distribution_channel', 'deposit_type', 'customer_type', 'guest_type', 'country_region']

df = pd.get_dummies(df, columns=dummy_cols, drop_first=True, dtype=int)
df.head()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,...,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,guest_type_Group,guest_type_Single,guest_type_Undefined,country_region_OTH,country_region_PRT
0,0,342,0,0,0,0,0,3,0,0.0,...,0,0,0,1,0,1,0,0,0,1
1,0,737,0,0,0,0,0,4,0,0.0,...,0,0,0,1,0,1,0,0,0,1
2,0,7,0,1,0,0,0,0,0,75.0,...,0,0,0,1,0,0,1,0,0,0
3,0,13,0,1,0,0,0,0,0,75.0,...,0,0,0,1,0,0,1,0,0,0
4,0,14,0,2,0,0,0,0,0,98.0,...,0,0,0,1,0,1,0,0,0,0


In [37]:
# 파일 저장
df.to_csv('../dataset/preprocessed/hotel_bookings_dummy.csv', index=False)